# 1 · Two agents and their GNAP login

Two agents of this data space, one in each role: the **provider** publishes data, the **consumer** negotiates for it and receives it. Any agent can take either role; the roles belong to this exchange, not to the agents.

The notebooks read who is who from the environment. Start them with the `notebook` action of an extension, the demo's for instance:

```bash
./eunomia demo notebook <provider> <consumer>      # e.g. ./eunomia demo notebook alice bob
```

which logs the two agents in with each other over GNAP, sets the variables below and opens JupyterLab on a session of that pair. This notebook is the same for every extension; the ones after it are the extension's.

In [ ]:
import json
import os

import requests

import helpers
from helpers import show, table, short

## Configuration

| Variable | What |
| --- | --- |
| `EUNOMIA_PROVIDER`, `EUNOMIA_CONSUMER` | the agents' names |
| `EUNOMIA_PROVIDER_URL`, `EUNOMIA_CONSUMER_URL` | their APIs, as this notebook reaches them |
| `EUNOMIA_CONSUMER_DOCKER_URL` | the consumer as the provider reaches it, from its container: the callback address in every DSP message |
| `EUNOMIA_SINK_*` | where pushed data lands: this session's own port |
| others | the extension's own: the demo adds `EUNOMIA_FACTORY_*`, the backend the provider shares |

Every call below is shown as a card: the route, the agent that answered, the HTTP status and time, the state it reached, the fields worth reading, and the request and response bodies (click to fold them). Long values such as DIDs are shortened on screen; the variables hold them in full.

In [ ]:
def env(name):
    try:
        return os.environ[name]
    except KeyError:
        raise RuntimeError(f"{name} is not set. Start the notebooks with an extension's notebook action: ./eunomia demo notebook <provider> <consumer>.") from None


provider_name = env("EUNOMIA_PROVIDER")
consumer_name = env("EUNOMIA_CONSUMER")
provider = env("EUNOMIA_PROVIDER_URL")
consumer = env("EUNOMIA_CONSUMER_URL")
consumer_docker = env("EUNOMIA_CONSUMER_DOCKER_URL")

json_header = {"Content-Type": "application/json"}

# How the cards name each agent.
helpers.AGENTS.update({provider: f"{provider_name} (provider)", consumer: f"{consumer_name} (consumer)"})

table(
    [
        {"role": "provider", "agent": provider_name, "api": provider, "console": provider + "/admin"},
        {"role": "consumer", "agent": consumer_name, "api": consumer, "console": consumer + "/admin"},
    ],
    [("Role", "role"), ("Agent", "agent"), ("API", "api"), ("Console", "console")],
    title="The two agents of this exchange",
);

## Who they are

Every agent has a DID. Locally it is a `did:jwk`, generated with the agent's own keys when the instance was created.

In [ ]:
response = requests.get(provider + "/api/v1/mates/myself", headers=json_header)
provider_did = show(response, f"{provider_name}'s own identity")["participant_id"]

In [ ]:
response = requests.get(consumer + "/api/v1/mates/myself", headers=json_header)
consumer_did = show(response, f"{consumer_name}'s own identity")["participant_id"]

## Their GNAP tokens

Before two agents trade, they log in with each other over GNAP: one asks the other for a grant (`POST /api/v1/peer-connection/connect`), the other verifies its credentials (membership and the Gaia-X label) and approves it, and both end up holding a token for the other.

The `notebook` action did that already; by hand it is `./eunomia login <provider> <consumer>`. Each agent keeps the participants it knows in `mates`, with the token it uses to talk to them.

In [ ]:
MATES = [
    ("Nick", "participant_nick"),
    ("Type", "participant_type"),
    ("Address", "base_url"),
    ("Token", lambda m: short(m.get("token") or "—", 24)),
    ("DID", lambda m: short(m["participant_id"], 40)),
]


def mates_of(agent, name):
    response = requests.get(agent + "/api/v1/mates/all", headers=json_header)
    mates = show(response, f"Who {name} knows", open_response=False)
    table([m for m in mates if not m.get("is_me")], MATES, title=f"{name}'s mates")
    return mates


provider_mates = mates_of(provider, provider_name)

In [ ]:
consumer_mates = mates_of(consumer, consumer_name)

In [ ]:
def token_for(mates, other_did):
    return next((m.get("token") for m in mates if m.get("participant_id") == other_did), None)


consumer_token = token_for(provider_mates, consumer_did)   # what the provider uses to talk to the consumer
provider_token = token_for(consumer_mates, provider_did)   # what the consumer uses to talk to the provider

if not (consumer_token and provider_token):
    raise RuntimeError(f"The two agents are not logged in with each other: ./eunomia login {provider_name} {consumer_name}")

table(
    [
        {"agent": provider_name, "talks to": consumer_name, "token": short(consumer_token, 24)},
        {"agent": consumer_name, "talks to": provider_name, "token": short(provider_token, 24)},
    ],
    [("Agent", "agent"), ("Talks to", "talks to"), ("GNAP token", "token")],
    title="Logged in with each other",
);